# C10-competition-craft — Practice p17

**Type:** challenge · **Difficulty:** advanced · **Budget:** 135 minutes

**Concepts:** writeup-quality, colab-markdown-solution-authoring, markdown-code-snippets, markdown-math-formulae, colab-coding-submission, cpu-and-gpu-round-boundary, train-test-split, f1-macro, knn, feature-scaling, sklearn-pipelines

Build the capped mini-competition as a deliberate mixed-cell Colab submission. Use `../data/train.csv` and the pinned carve `test_size=150, random_state=SEED, stratify=y`. The baseline is exactly a scaled 5-NN pipeline on all 12 features. Log its validation macro-F1 before any selection.

Then perform exactly three moves: (1) sweep `k` over `[5, 7, 9, 11, 15]` on all 12 features, smallest `k` on ties; (2) try the seven `SIGNAL` features at the current `k` and keep them only if macro-F1 strictly improves; (3) repeat the same `k` sweep on the current feature set, again choosing the smallest `k` on ties. No extra candidates or re-carves.

Submit one artifact for each of the **six separately scored rows**.

The solution preserves the exact four-row log semantics, deterministic two-run equality, full-data refit, stage trace, and the pinned `p17_log.csv` and `p17_predictions.csv` artifacts. The fenced function is communication only, while the identical executable definition is in a code cell. Round 1 remains CPU-only.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804
SIGNAL = ["honey_stores_kg", "autumn_hive_mass_kg", "varroa_mite_index",
          "forager_traffic_per_min", "brood_frames", "daily_temp_swing_c",
          "queen_age_years"]
sweep_ks = np.array([5, 7, 9, 11, 15])

df = pd.read_csv("../data/train.csv")
FEATURES = [c for c in df.columns if c != "outcome"]
X = df[FEATURES]
y = df["outcome"].to_numpy()
stage_trace = ["setup"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=150, random_state=SEED, stratify=y
)

def fitted_score(feature_names, k):
    model = Pipeline([
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=int(k))),
    ])
    model.fit(X_train[feature_names], y_train)
    score = f1_score(y_val, model.predict(X_val[feature_names]), average="macro")
    return float(score)

In [ ]:
changes = [
    "scaled 5-NN, all 12 features",
    "k sweep [5, 7, 9, 11, 15]",
    "try SIGNAL features",
    "repeat k sweep [5, 7, 9, 11, 15]",
]
log_rows = []

selected_features = FEATURES.copy()
selected_k = 5
current_score = fitted_score(selected_features, selected_k)
log_rows.append({"step": "baseline", "change": changes[0],
                 "val_f1": current_score, "accepted": True})
stage_trace.append("baseline")

first_sweep = [(int(k), fitted_score(FEATURES, int(k))) for k in sweep_ks]
first_k, first_score = max(first_sweep, key=lambda item: (item[1], -item[0]))
log_rows.append({"step": "iter-1", "change": changes[1],
                 "val_f1": first_score, "accepted": first_k != selected_k})
selected_k, current_score = first_k, first_score
stage_trace.append("iter-1")

signal_score = fitted_score(SIGNAL, selected_k)
signal_accepted = signal_score > current_score
log_rows.append({"step": "iter-2", "change": changes[2],
                 "val_f1": signal_score, "accepted": signal_accepted})
if signal_accepted:
    selected_features = SIGNAL.copy()
    current_score = signal_score
stage_trace.append("iter-2")

third_sweep = [(int(k), fitted_score(selected_features, int(k))) for k in sweep_ks]
third_k, third_score = max(third_sweep, key=lambda item: (item[1], -item[0]))
log_rows.append({"step": "iter-3", "change": changes[3],
                 "val_f1": third_score, "accepted": third_k != selected_k})
selected_k, current_score = third_k, third_score
stage_trace.append("iter-3")

log_df = pd.DataFrame(log_rows, columns=["step", "change", "val_f1", "accepted"])
final_val_f1 = float(current_score)

In [ ]:
final_model = Pipeline([
    ("scale", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=selected_k)),
])
final_model.fit(X[selected_features], y)

def predict_labels(X_test):
    predictions = final_model.predict(X_test[selected_features])
    return pd.Series(predictions, index=X_test.index)

stage_trace.append("refit")
probe = X.iloc[150:190]
probe_predictions = predict_labels(probe)
contract_checks = {
    "series": isinstance(probe_predictions, pd.Series),
    "length": len(probe_predictions) == len(probe),
    "index": probe_predictions.index.equals(probe.index),
    "vocab": set(probe_predictions.unique()) <= set(np.unique(y)),
}
assert all(contract_checks.values())
contract_ok = all(contract_checks.values())

In [ ]:
def run_submission():
    X_tr, X_va, y_tr, y_va = train_test_split(
        X, y, test_size=150, random_state=SEED, stratify=y
    )

    def evaluate(feature_names, k):
        candidate = Pipeline([
            ("scale", StandardScaler()),
            ("knn", KNeighborsClassifier(n_neighbors=int(k))),
        ])
        candidate.fit(X_tr[feature_names], y_tr)
        value = f1_score(y_va, candidate.predict(X_va[feature_names]), average="macro")
        return float(value)

    rows = []
    features = FEATURES.copy()
    k_now = 5
    score_now = evaluate(features, k_now)
    rows.append({"step": "baseline", "change": changes[0],
                 "val_f1": score_now, "accepted": True})

    sweep_one = [(int(k), evaluate(FEATURES, int(k))) for k in sweep_ks]
    k_one, score_one = max(sweep_one, key=lambda item: (item[1], -item[0]))
    rows.append({"step": "iter-1", "change": changes[1],
                 "val_f1": score_one, "accepted": k_one != k_now})
    k_now, score_now = k_one, score_one

    score_signal = evaluate(SIGNAL, k_now)
    accept_signal = score_signal > score_now
    rows.append({"step": "iter-2", "change": changes[2],
                 "val_f1": score_signal, "accepted": accept_signal})
    if accept_signal:
        features = SIGNAL.copy()
        score_now = score_signal

    sweep_three = [(int(k), evaluate(features, int(k))) for k in sweep_ks]
    k_three, score_three = max(sweep_three, key=lambda item: (item[1], -item[0]))
    rows.append({"step": "iter-3", "change": changes[3],
                 "val_f1": score_three, "accepted": k_three != k_now})
    k_now, score_now = k_three, score_three

    rebuilt = Pipeline([
        ("scale", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k_now)),
    ])
    rebuilt.fit(X[features], y)
    probe_local = X.iloc[150:190]
    pred = pd.Series(rebuilt.predict(probe_local[features]), index=probe_local.index)
    log_copy = pd.DataFrame(rows, columns=["step", "change", "val_f1", "accepted"])
    return log_copy, float(score_now), pred

run_a = run_submission()
run_b = run_submission()
assert run_a[0].equals(run_b[0])
assert run_a[1] == run_b[1]
assert run_a[2].equals(run_b[2])
cell_plan = [
    ("text", "Approach, Intuition, Alternatives, and limitation"),
    ("code", "imports, pinned carve, baseline, and three bounded moves"),
    ("code", "full-data refit and executable predict_labels contract"),
    ("text", "matching fenced predict_labels excerpt"),
    ("text", "macro-F1 display formula and equal-weight interpretation"),
    ("code", "independent two-run equality check and exact CSV packaging"),
    ("text", "Round 1 CPU and Round 2 L4/GPU declaration"),
]

In [ ]:
log_df.to_csv("p17_log.csv", index=False)
pd.DataFrame({"row_id": probe.index,
              "prediction": probe_predictions.to_numpy()}).to_csv(
    "p17_predictions.csv", index=False
)
stage_trace.append("package")

saved_log = pd.read_csv("p17_log.csv")
saved_predictions = pd.read_csv("p17_predictions.csv")
assert list(saved_log.columns) == ["step", "change", "val_f1", "accepted"]
assert saved_log["step"].tolist() == ["baseline", "iter-1", "iter-2", "iter-3"]
assert saved_log["change"].tolist() == log_df["change"].tolist()
assert np.allclose(saved_log["val_f1"], log_df["val_f1"], atol=1e-12, rtol=0)
assert saved_log["accepted"].tolist() == log_df["accepted"].tolist()
assert list(saved_predictions.columns) == ["row_id", "prediction"]
assert len(saved_predictions) == 40
assert saved_predictions["row_id"].tolist() == probe.index.tolist()
assert saved_predictions["prediction"].tolist() == probe_predictions.tolist()
assert stage_trace == ["setup", "baseline", "iter-1", "iter-2", "iter-3", "refit", "package"]
assert np.isclose(final_val_f1, 0.8198760747394207, atol=1e-12, rtol=0)

## Row 1 — Mini-competition writeup

### Approach

Using the single pinned stratified carve (`test_size=150`, `random_state=20260804`), the baseline scaled 5-NN on all 12 features scored macro-F1 `0.7665823769694612`. The first bounded sweep selected `k=11` on all features, the strict-improvement feature move kept the seven `SIGNAL` features, and the repeated sweep retained `k=11`. The selected recipe is therefore a scaled 11-NN pipeline on `SIGNAL`, with `final_val_f1 = 0.8198760747394207`; it is refit on all 600 labeled rows before prediction.

### Intuition

Scaling prevents large-unit features from dominating Euclidean neighbor distances. The bounded neighbor sweep balances noisy small neighborhoods against overly smooth large ones, while macro-F1 gives equal influence to both classes in the 2:1 task.

### Alternatives

A logged alternative was scaled 11-NN on all 12 features at `0.8103481812876873`; the `SIGNAL` candidate at the same `k` improved it to `0.8198760747394207`. Because all three moves select on the same validation carve, this score can be optimistic and is not an untouched-test estimate.

## Row 3 — Fenced `predict_labels` excerpt

```python
def predict_labels(X_test):
    predictions = final_model.predict(X_test[selected_features])
    return pd.Series(predictions, index=X_test.index)
```

## Row 4 — Rendered metric

For each class $k$, let $F_{1,k}$ be that class's harmonic mean of precision and recall. For $K$ classes,

$$
F_{1,\mathrm{macro}} = \frac{1}{K} \sum_{k=1}^{K} F_{1,k}.
$$

Here $K=2$. Equal class weight fits the 2:1 task because performance on the smaller class cannot be hidden by the larger class's count.

## Row 6 — Round declaration

**Round 1 is CPU-only; Round 2 permits Colab L4/GPU.**

### Answer check

The bounded four-row log, selected scaled 11-NN `SIGNAL` recipe, two-run equality, full-data refit, prediction contract, exact CSV artifacts, and seven-stage trace are all checked; floating comparisons use `atol=1e-12` and `rtol=0`.